In [22]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://finance.yahoo.com/markets/crypto/most-active/"

HEADERS = {'User-Agent': 'Mozilla/5.0'}

def parse_human_readable(value):
    if not value:
        return None
    val = value.replace(',', '')
    last_char = val[-1]
    val_no_last_char = val[:-1]

    match (last_char):
        case 'K':
            return float(val_no_last_char) * 1000
        case 'M':
            return float(val_no_last_char) * 1_000_000
        case 'B':
            return float(val_no_last_char) * 1_000_000_000
        case 'T':
            return float(val_no_last_char) * 1_000_000_000_000
        case _:
            return float(val)


def crypto_volumes():
    response = requests.get(URL, headers=HEADERS)
    print(response)
    print(response.status_code)
    html = response.text
    #print(html)
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find('table', attrs={'class': 'yf-19xszjc'})
    #print(table)
    rows = table.find('tbody').find_all('tr')
    data = []
    for row in rows:
        #print(row)
        #print('------------------------')
        cells = row.find_all('td')
        symbol = cells[0].text.strip()
        #print(symbol)
        price = cells[4].text.strip()
        market_cap = cells[6].text.strip()
        volume = cells[8].text.strip()
        record = {
            'symbol': symbol,
            'price': parse_human_readable(price),
            'market_cap': market_cap,
            'volume': volume,
        }
        #print(record)
        data.append(record)
    df = pd.DataFrame(data)
    df['market_cap'] = df['market_cap'].apply(lambda x: parse_human_readable(x))
    df['volume'] = df['volume'].apply(parse_human_readable)
    return df


def save_crypto_volumes(df):
    df.to_csv('data/crypto_volumes.csv', index=False)
    df.to_json('data/crypto_volumes.json', index=False)
    df.to_excel('data/crypto_volumes.xlsx', index=False)

In [20]:
def x2(x):
    return x*2

print(x2(1000))

x_2 = lambda x: x*2
print(x_2(1000))

2000
2000


In [23]:
df = crypto_volumes()
save_crypto_volumes(df)

<Response [200]>
200


ModuleNotFoundError: No module named 'openpyxl'